# 31. How much unlearning is enough?

Notebook 30 found that unlearning turns the weights as well as shrinking them, and that the
turning is late. At step 20 the weights had shrunk by a fifth and turned by 7 degrees. At
step 100 they had turned 49 degrees.

A rotation cannot be undone by rescaling. So if the removed task already sits at chance
accuracy by step 20 or 40, the steps after that are not buying forgetting. They are buying
irreversibility, and the model at the earlier step would be both forgotten and recoverable.

That is the privacy question, and it needs no new run. Notebook 27 recorded the removed
task's accuracy every 20 steps and saved it. Notebook 30 recorded the direction at the same
steps. This notebook joins them on the step number and prints one table.

**No GPU, no images, no model.** It downloads two small files and does arithmetic.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/path-join")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
if importlib.util.find_spec("boto3") is None:
    if not IN_COLAB:
        raise ImportError("Install boto3 first.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3"], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)

## 2. Settings

These must match the run being joined. Both notebooks recorded their own seed and task, and
the cells below refuse to join files that disagree with these values, so a mismatch stops the
notebook rather than producing a table that silently mixes two runs.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEED = 0
FORGET_TASK = "3"

# Ten classes per task, so a model that has lost the task sits here.
CHANCE_PCT = 10.0
# Accuracy within this much of chance counts as forgotten. The removed task cannot go
# meaningfully below chance, so this is a one-sided margin.
FORGOTTEN_WITHIN_PCT = 2.0
# Above this cosine we treat the weights as still pointing the same way, so a rescaling has
# a chance of undoing the change. 0.95 is about 18 degrees.
RECOVERABLE_ABOVE = 0.95

CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print(f"seed {SEED}, forgetting task {FORGET_TASK}, chance is {CHANCE_PCT}%")

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
PREFIX = "uncle/paired_generalization/" + STUDY_ID
print("Connected to", BUCKET)

## 4. Find notebook 27's run

Notebook 27 writes one folder per run, named with the time it finished, so there may be
several. We keep the ones that match this seed and task, and use the most recent. If more
than one matches, the cell says so, because two runs of the same thing are worth knowing
about before their numbers are used.

In [ ]:
# Every diagnostics folder notebook 27 has ever written.
candidates = sorted({key.rsplit("/", 1)[0] for key in store.keys(PREFIX + "/diagnostics/")
                     if key.endswith("/summary.json")})
print(f"Found {len(candidates)} diagnostics folders")

matching = []
for folder in candidates:
    try:
        summary = json.loads(store.download(folder + "/summary.json", CACHE / "cache").read_text())
    except Exception as problem:
        print("  unreadable:", folder, problem)
        continue
    if summary.get("notebook") != "27_weight_change_and_sensitivity":
        continue
    if (summary.get("seed") == SEED and summary.get("forget_task") == FORGET_TASK
            and summary.get("order") == ORDER):
        matching.append((folder, summary))
        print(f"  match: {folder.rsplit('/', 1)[-1]}  "
              f"{summary['steps']} steps, interval {summary['interval']}, "
              f"device {summary['environment']['device']}")

if not matching:
    raise RuntimeError(f"No notebook 27 run found for seed {SEED}, task {FORGET_TASK}.")
if len(matching) > 1:
    print(f"\n{len(matching)} runs match. Using the most recent. The others are listed above.")
FOLDER, SUMMARY = matching[-1]
ACCURACY_ROWS = {row["step"]: row for row in SUMMARY["readings"]}
print(f"\nUsing {FOLDER.rsplit('/', 1)[-1]} with {len(ACCURACY_ROWS)} readings")

## 5. Load notebook 30's result

In [ ]:
DIRECTION_KEY = f"uncle/unlearning_direction/{STUDY_ID}/{ORDER}/forget{FORGET_TASK}.json"
direction_file = json.loads(store.download(DIRECTION_KEY, CACHE / "cache").read_text())
run = next((r for r in direction_file["runs"] if r["seed"] == SEED), None)
if run is None:
    raise RuntimeError(f"Notebook 30 has no seed {SEED} in {DIRECTION_KEY}.")
if direction_file["forget_task"] != FORGET_TASK:
    raise RuntimeError("The direction file is for a different forgotten task.")
DIRECTION_ROWS = {row["step"]: row for row in run["rows"]}
print(f"Direction readings at steps: {sorted(DIRECTION_ROWS)}")
print(f"Accuracy readings at steps:  {sorted(ACCURACY_ROWS)}")
SHARED = sorted(set(DIRECTION_ROWS) & set(ACCURACY_ROWS))
if not SHARED:
    raise RuntimeError("The two runs share no step numbers, so they cannot be joined.")
print("Joining on:", SHARED)

## 6. Are the two runs the same event?

Both notebooks record `weight_size`, the size of the generator's raw output for the forgotten
task. It is the same quantity computed the same way, so if the two runs did the same thing it
should agree at every shared step.

It may not agree exactly. Notebook 27 ran on a GPU and notebook 30 on a CPU, and small
floating point differences compound over a hundred optimisation steps. A small and slowly
growing gap is expected. A large gap means the two runs diverged and the joined table below
would be comparing different things.

In [ ]:
print(f"{'step':>5} {'size (nb27)':>13} {'size (nb30)':>13} {'gap':>9} {'gap %':>8}")
worst = 0.0
for step in SHARED:
    first = ACCURACY_ROWS[step]["weight_size"]
    second = DIRECTION_ROWS[step]["forgotten"]["raw_weight_size"]
    gap = second - first
    share = 100.0 * abs(gap) / first if first else float("nan")
    worst = max(worst, share)
    print(f"{step:>5} {first:>13.4f} {second:>13.4f} {gap:>9.4f} {share:>7.3f}%")

SAME_EVENT = worst < 1.0
print()
if SAME_EVENT:
    print(f"Largest gap {worst:.3f}%. The two runs track each other; the join is sound.")
else:
    print(f"Largest gap {worst:.3f}%. That is too large to treat these as one run.")
    print("Read the table below as two separate runs placed side by side, not as a join.")

## 7. Forgetting against direction

One row per reading. The question is where the accuracy column reaches chance, and what the
direction column says at that same step.

In [ ]:
import math

print(f"{'step':>5} | {'removed acc %':>13} {'classes used':>12} {'sensitivity':>12} "
      f"| {'direction':>9} {'turned':>8} {'size chg':>9}")
first_forgotten = None
for step in SHARED:
    accuracy_row = ACCURACY_ROWS[step]
    forgotten = DIRECTION_ROWS[step]["forgotten"]
    accuracy = accuracy_row["removed_task_accuracy_pct"]
    cosine = forgotten["direction"]
    # Cosine is hard to read as a size. Degrees are not.
    turned = math.degrees(math.acos(max(-1.0, min(1.0, cosine))))
    if first_forgotten is None and accuracy <= CHANCE_PCT + FORGOTTEN_WITHIN_PCT:
        first_forgotten = step
    print(f"{step:>5} | {accuracy:>13.2f} {accuracy_row['removed_task_distinct_classes']:>12} "
          f"{accuracy_row['input_sensitivity']:>12.3e} | {cosine:>9.4f} "
          f"{turned:>7.1f}d {forgotten['relative_change']:>9.4f}")

print()
if first_forgotten is None:
    print(f"The removed task never reached chance ({CHANCE_PCT}%) at any reading.")
    print("Either the unlearning is incomplete, or the readings are too far apart to see it.")
else:
    at_forgotten = DIRECTION_ROWS[first_forgotten]["forgotten"]["direction"]
    final_step = SHARED[-1]
    at_end = DIRECTION_ROWS[final_step]["forgotten"]["direction"]
    turned_then = math.degrees(math.acos(max(-1.0, min(1.0, at_forgotten))))
    turned_end = math.degrees(math.acos(max(-1.0, min(1.0, at_end))))
    print(f"First reading at chance: step {first_forgotten}, where the weights had turned "
          f"{turned_then:.1f} degrees.")
    print(f"By step {final_step} they had turned {turned_end:.1f} degrees.")
    print()
    if at_forgotten >= RECOVERABLE_ABOVE:
        print(f"At step {first_forgotten} the direction is still {at_forgotten:.4f}, above the "
              f"{RECOVERABLE_ABOVE} mark.")
        print("So the task is already gone at a point where the weights have barely turned.")
        print(f"The {final_step - first_forgotten} steps after that turn the weights without "
              "removing more accuracy.")
        print("That is worth testing directly: unlearn only to that step and try to recover.")
    else:
        print(f"At step {first_forgotten} the direction is already {at_forgotten:.4f}, below the "
              f"{RECOVERABLE_ABOVE} mark.")
        print("The weights had turned before the accuracy reached chance, so there is no")
        print("obvious earlier stopping point that both forgets and stays recoverable.")
    print()
    print("The readings are", SHARED[1] - SHARED[0], "steps apart, so the true crossing point")
    print("lies somewhere in the interval before this step. A finer interval would locate it.")

## What to take from this

**Read the accuracy column first.** The removed task cannot fall meaningfully below chance,
which is 10 percent for ten classes. The step where it first arrives there is the point at
which the task is gone.

**Then read the direction at that same step.** If the weights have barely turned there, the
unlearning reached its goal while still being a near-scaling, and everything after is turning
the weights for no extra forgetting. If they had already turned, there is no clean earlier
stopping point.

**The classes used column is a check on the accuracy.** A model that has genuinely lost a
task spreads its guesses. A model that answers the same class for every image also scores
near chance but for a different reason, and that shows up as a single class being used.

**Input sensitivity says whether the model still reacts to its input at all.** It falls to
near zero when the generated network outputs almost the same scores whatever it is shown.
That is the collapse the 21 October record describes, and it is a stronger statement than
low accuracy.

**The crossing point is bracketed, not located.** Readings are 20 steps apart, so the first
reading at chance only says the crossing happened in the previous 20 steps. Narrowing it
needs a run with a smaller interval, which is cheap on CPU.

**One seed.** Everything here describes seed 0. Say so when reporting it.